# Tier 2 — the Natural branch: where will lightning fire burn?

One of the three Tier-2 branch deep-dives off the Tier-1 coarse allocator
([`06_analysis.ipynb`](06_analysis.ipynb)). Tier 1 established that **Natural (lightning) is 59% of
all burned area** — and it is a cause the planner *cannot prevent*. The only lever is **mitigation**
(fuel treatment, defensible space, suppression pre-positioning), and mitigation is about **where**
the burn lands, not **what** ignited it. So unlike the Human branch, this is not a cause-composition
problem — Natural is effectively one cause — it is a **location** problem: *which region-seasons will
carry the Natural burn next season?*

**Deliverable.** A forward-looking map of expected Natural burned area across region-seasons, so
mitigation effort concentrates where the acres are going to be. First pass at **EPA Level III**
grain; whether a finer grain is needed is deferred by design, not assumed here.

**Scope note.** Per the design, the Natural branch is a methodologically distinct sub-project from
the RQ2 forecast (which is Tier 1 + the Human branch). It tells the planner what the cause-risk
profile *cannot* act on by prevention.

**Method.** Same grain, partial-winter boundary rule, and forward-chaining split as the rest of the
pipeline. Three products: (1) a **descriptive concentration** read — how concentrated Natural burn is
across regions within a season — (2) a **predictive first pass** on per-region-season Natural
acres, scored log-space (the same heavy-tailed rationale as the Tier-1 level) against a persistence
floor, and (3) an **external-covariate rung** built on pre-season drought and fuel-dryness data
(TerraClimate), which is the first activation of the climate/fuels layer deferred since W2. That
layer is here because the persistence floor fails in a specific, diagnosed way: it under-predicts
every megafire cell by 1–1.7 orders of magnitude, so a big lightning year is a weather event the
fire record alone cannot anticipate.

**Conventions.** Cells run top-to-bottom, left **unexecuted** for manual run.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
from config import ProjectConfig

# Project-wide constants (paths, the boundary rule, the forward-chaining split) come
# from one place so this branch cannot drift from the others. See src/config.py.
cfg = ProjectConfig()
DATA = cfg.data

rsc = pd.read_parquet(cfg.region_season_cause)

# Same boundary rule as every other branch: drop the two structurally-truncated DJF seasons.
rsc = rsc[~rsc["season_idx"].isin(cfg.partial_winters)].copy()

# Natural = the single 'Natural' cause. One row per region-season with its Natural burned acres.
KEYS = list(cfg.cell_keys)
nat = (rsc[rsc["cause"] == "Natural"].groupby(KEYS, observed=True)["acres"].sum()
       .rename("nat_ac").reset_index())
nat = nat.sort_values(list(cfg.sort_keys)).reset_index(drop=True)

print(f"{len(nat):,} region-season cells | {(nat.nat_ac > 0).sum():,} with Natural burn > 0")
print(f"per-cell Natural acres: median {nat.loc[nat.nat_ac>0,'nat_ac'].median():,.0f}, "
      f"max {nat.nat_ac.max():,.0f}  (heavy-tailed -> log-space scoring)")

## Descriptive: how concentrated is Natural burn?

Before predicting *where*, quantify how spatially concentrated Natural burn is — this is what makes
targeted mitigation worthwhile at all. For each season-year, two concentration measures across
regions: the **Herfindahl index (HHI)** of Natural acres (1 = one region has it all; ~0 = spread
evenly) and the **top-region share** (what fraction the single biggest region carries). If Natural
burn were spread uniformly, mitigation siting would have nothing to target; concentration is the
premise.

In [2]:
def hhi(x):
    s = x.sum()
    return np.nan if s <= 0 else float(((x / s) ** 2).sum())

def top_share(x):
    s = x.sum()
    return np.nan if s <= 0 else float(x.max() / s)

pos = nat[nat["nat_ac"] > 0]
conc = pos.groupby(["season", "season_year"])["nat_ac"].agg(HHI=hhi, top_share=top_share)

by_season = conc.groupby("season").mean()
by_season["n_regions_burning"] = pos.groupby(["season", "season_year"]).size().groupby("season").mean()
print("Natural-burn concentration across regions, mean over years, by season:\n")
print(by_season.round(3).to_string())
print("\nSummer (JJA) spreads across many western regions (low HHI); winter/spring concentrate in a")
print("few (high HHI, top region ~half) -- so where to mitigate is itself season-dependent.")

Natural-burn concentration across regions, mean over years, by season:

          HHI  top_share  n_regions_burning
season                                     
DJF     0.380      0.510             23.679
JJA     0.137      0.262             81.655
MAM     0.345      0.497             64.793
SON     0.248      0.392             60.414

Summer (JJA) spreads across many western regions (low HHI); winter/spring concentrate in a
few (high HHI, top region ~half) -- so where to mitigate is itself season-dependent.


## Predictive first pass: next-season Natural acres per region

The location product: predict each region-season's Natural burned acres, so the biggest expected
burdens can be ranked for mitigation. Same machinery as the Tier-1 level — predict `log10(nat_ac)`
with a forward-chaining trailing mean, scored on the held-out tail (season-year ≥ 2010) with
log-MAE, reported both unweighted and **acre-weighted** ("typically off by a factor of ~N"). Cells
with zero Natural burn are excluded from the log target (many non-western cells simply never see
lightning fire); the prediction problem is *how much*, given that a region burns at all.

We compare persistence against a deliberately dumb **global prior** (one constant for every cell).
Watch for the two metrics to *disagree* — that split is itself the branch's headline result, and it
is what motivates the external drought/fuel-dryness covariates that follow.

In [ ]:
TEST_START = cfg.test_start     # forward-chaining split, shared across all branches
K = cfg.shares_k                # trailing window locked by the Tier-1 shares sweep

# The forward-chaining rule lives in src/trailing.py rather than being re-typed per
# notebook: TrailingMean does shift(1) then a k-window mean within (region, season)
# and asserts the frame is sorted first. GlobalPrior is the deliberately uninformed
# reference -- one constant everywhere, fit on training years only.
from trailing import GlobalPrior, TrailingMean

natp = nat[nat["nat_ac"] > 0].copy()
natp["log_nat"] = np.log10(natp["nat_ac"])
natp = natp.sort_values(list(cfg.sort_keys)).reset_index(drop=True)

actual = natp["log_nat"].to_numpy()
w = natp["nat_ac"].to_numpy()                           # acre weight (rank the big burdens)
in_test = (natp["season_year"] >= TEST_START).to_numpy()
train = (natp["season_year"] < TEST_START).to_numpy()

def logmae(pred):
    err = np.abs(pred - actual)
    m = in_test & ~np.isnan(err)
    return {"n_cells": int(m.sum()),
            "logMAE_unwtd": float(err[m].mean()),
            "logMAE_acre_wtd": float(np.average(err[m], weights=w[m])),
            "x_off_acre_wtd": float(10 ** np.average(err[m], weights=w[m]))}

pred_trail = TrailingMean(K).predict(natp, "log_nat")["log_nat"].to_numpy()

# Reference: a global 'typical Natural acres' prior from training years only (no region
# info). Acre-weighted, so it reflects where the acres actually are.
prior = GlobalPrior(weighted=True).fit(natp, "log_nat", train_mask=train, weight_col="nat_ac")
global_log = float(prior.value_[0])
pred_global = prior.predict(natp)["log_nat"].to_numpy()

res = pd.DataFrame([logmae(pred_global), logmae(pred_trail)],
                   index=["global prior (train mean)", f"persistence (k={K})"])
print(f"NATURAL location -- log-space, held-out tail season_year >= {TEST_START}\n")
print(res.round(4).to_string())

print("\nRead this split carefully -- the two metrics disagree, and the disagreement is the finding:")
print("  * UNWEIGHTED: persistence wins big (it nails the many small/typical cells).")
print("  * ACRE-WEIGHTED: the global prior wins -- because on the megafire cells that carry the")
print("    acres, a region's own calm-year history badly UNDER-predicts its record year, while a")
print("    high global constant lands closer. History is actively misleading on the big burns.")

In [4]:
# The evidence for that reading: the biggest held-out Natural cells, and how each predictor does.
m = in_test & ~np.isnan(pred_trail)
diag = natp[m].copy()
diag["persist_log"] = pred_trail[m]
diag["global_log"] = global_log
diag["err_persist"] = np.abs(diag["persist_log"] - diag["log_nat"])
diag["err_global"] = np.abs(diag["global_log"] - diag["log_nat"])

top = diag.nlargest(6, "nat_ac")[
    ["region", "season", "season_year", "nat_ac", "log_nat",
     "persist_log", "global_log", "err_persist", "err_global"]]
print(f"global prior = {global_log:.2f} log-acres (~{10**global_log:,.0f} ac); "
      f"test-cell median = {np.median(actual[m]):.2f} log-acres (~{10**np.median(actual[m]):,.0f} ac)\n")
print("The six largest held-out Natural burns -- persistence under-predicts every one by 1-1.7 orders:")
print(top.round(2).to_string(index=False))
print("\nThis is the Tier-1 'can't see a megafire coming' result in its sharpest form, and the case")
print("for external pre-season covariates: the acres live in years history cannot anticipate.")

global prior = 5.46 log-acres (~285,412 ac); test-cell median = 1.55 log-acres (~35 ac)

The six largest held-out Natural burns -- persistence under-predicts every one by 1-1.7 orders:
                                             region season  season_year     nat_ac  log_nat  persist_log  global_log  err_persist  err_global
             Interior Forested Lowlands and Uplands    JJA         2015 2826326.39     6.45         4.73        5.46         1.72        1.00
                           Northern Basin and Range    JJA         2012 2115903.79     6.33         5.17        5.46         1.15        0.87
                               Interior Bottomlands    JJA         2015 1278711.00     6.11         4.60        5.46         1.50        0.65
Klamath Mountains/California High North Coast Range    JJA         2020 1176272.43     6.07         4.68        5.46         1.39        0.62
             Interior Forested Lowlands and Uplands    JJA         2019 1161385.00     6.06         5.30 

## External covariates — drought & fuel dryness (deferred layer, now live)

The persistence floor above predicts a region's next-season Natural acres from its own past. But a
big lightning-fire year is driven by *that season's* dryness and fuel state, which history cannot
see — the same ceiling the Tier-1 level hit. The diagnostic above is the case for going outside the
fire record: on the six largest held-out cells, persistence under-predicts **every one** by 1–1.7
orders of magnitude.

This is no longer a stub. The covariates are **sourced, fetched, and joined** — see
[`../src/terraclimate.py`](../src/terraclimate.py) for the loader and
`data/region_season_climate.parquet` for the artifact.

**Source: TerraClimate** (Climatology Lab, U. Idaho), 1/24° (~4 km) monthly grids, 1958–present,
pulled via THREDDS/OPeNDAP with a server-side bounding box.

**Why this source and not the obvious ones.** The analysis grain includes **20 Alaska** Level III
ecoregions, and that single fact eliminated both standard drought products: **gridMET PDSI** and
**nClimGrid SPEI/PDSI** are CONUS-only, so either would have silently dropped every AK cell.
TerraClimate is global terrestrial — full panel coverage on both landmasses.

**Why fuel *condition*, not fuel *load*.** LANDFIRE was the candidate for fuel load and was
**rejected for this panel**: its base map is circa 2001 with discrete vintages
(2001/2008/2010/2012/2014/2016/2020) and usable Alaska coverage only from the 2016 Remap. Against a
1992–2020 season-year panel it contributes almost no *interannual* variance — precisely the variance
the megafire-year problem needs explained. It would have been a near-constant per region. So the
fuel signal here is dryness: `def`, `soil`, `vpd` describe how dry the fuel *is* going into the
season, which drives a big lightning year far more than how much fuel exists. Fuel load is a slow
cross-region story that ecoregion identity already partly encodes.

| Covariate | TerraClimate var | What it carries |
| --- | --- | --- |
| `pdsi` | `PDSI` | Palmer Drought Severity Index — the direct drought ask |
| `soil_moisture` | `soil` | column soil moisture (mm) — antecedent wetness |
| `water_deficit` | `def` | climatic water deficit (mm) — dryness with an energy term |
| `vpd` | `vpd` | vapor pressure deficit (kPa) — atmospheric dryness driver |

**Leakage rule — the part that matters most.** Each cell value is the mean over the **3 calendar
months immediately preceding the target season's first month**. Nothing from within the target
season is ever read. The winter case is the trap and is handled explicitly: DJF of year *Y* begins
in **December of *Y−1***, so its pre-season window is Sep–Nov of *Y−1*, not of *Y*. Getting that
wrong would leak a full year of future climate into every winter cell. The rule is asserted in
`terraclimate._self_check()` and re-verified in the notebook below before any model sees the data.

Spatial aggregation is an **area-weighted** mean over grid centers inside each ecoregion polygon,
weighted by cos(latitude) so the convergence of meridians does not over-weight northern cells —
material at Alaska's latitudes.

In [ ]:
import sys
sys.path.insert(0, str(Path("..") / "src"))
import terraclimate as tc

# Re-assert the leakage rule here, in the notebook, rather than trusting the module.
# This is the highest-risk seam in the branch: an off-by-one on the DJF year boundary
# would leak 12 months of future climate into every winter cell.
tc._self_check()

clim = pd.read_parquet(DATA / "region_season_climate.parquet")
COVS = ["pdsi", "soil_moisture", "water_deficit", "vpd"]

print(f"\nclimate table: {clim.shape[0]:,} region-season cells x {len(COVS)} covariates")
print(f"regions {clim['region'].nunique()} | season_year {clim['season_year'].min()}-{clim['season_year'].max()}")
print(f"missing per covariate:\n{clim[COVS].isna().sum().to_string()}")

# Join onto the modeling frame. Left join: every natp cell keeps its row, and a covariate
# that failed to resolve shows up as NaN rather than silently dropping a fire cell.
natx = natp.merge(clim[["region", "season", "season_year"] + COVS],
                  on=["region", "season", "season_year"], how="left")
assert len(natx) == len(natp), "join changed row count -- key collision"

matched = natx[COVS].notna().all(axis=1)
print(f"\njoined: {matched.sum():,}/{len(natx):,} cells have all four covariates "
      f"({matched.mean():.1%})")

### The learned rung: do the covariates beat the floor?

The covariates enter on **exactly the same terms** as the baselines above — same cells, same
forward-chaining split (`season_year >= 2010` held out), same log-MAE reported both unweighted and
acre-weighted. Nothing else changes, so any movement is attributable to the external data alone.

Two rungs, in ablation order, so added complexity has to earn its place:

1. **climate only** — the four covariates, no fire history. Tests whether pre-season dryness alone
   carries the signal.
2. **persistence + climate** — the trailing log-acres feature plus the covariates. Tests whether
   the external data adds anything *on top of* what history already knew.

A gradient-boosted tree (not linear) because the expected relationship is threshold-like: burned
area does not respond linearly to PDSI, it responds once a region crosses into drought. The model
is fit on training years only and never sees a held-out row.

**The honest read to watch for.** The floor's failure was specifically on the **acre-weighted**
metric — the megafire cells. So the question is not "does log-MAE improve on average" but "does the
acre-weighted error come down." An improvement only in the unweighted metric would mean the
covariates sharpened the many small cells and left the actual problem untouched.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

# The scoring frame is natx (natp + covariates), so `actual`, `w` and `in_test` from the
# baseline cell still align row-for-row -- same cells, same order, same split.
assert (natx["log_nat"].to_numpy() == actual).all(), "row alignment broke"

trail_feat = pred_trail                       # the persistence prediction, reused as a feature
X_clim = natx[COVS].to_numpy()
X_both = np.column_stack([trail_feat, X_clim])

def fit_score(X, label):
    """Fit on training years, score the held-out tail on the same metric as the floor."""
    ok = ~np.isnan(X).any(axis=1)
    tr = train & ok
    model = HistGradientBoostingRegressor(
        max_depth=3, max_iter=300, learning_rate=0.05, random_state=0)
    model.fit(X[tr], actual[tr])
    pred = np.full(len(actual), np.nan)
    pred[ok] = model.predict(X[ok])
    return logmae(pred) | {"model": label}

rows = [
    logmae(pred_global) | {"model": "global prior (train mean)"},
    logmae(pred_trail) | {"model": f"persistence (k={K})"},
    fit_score(X_clim, "climate only (4 covariates)"),
    fit_score(X_both, "persistence + climate"),
]
ladder = pd.DataFrame(rows).set_index("model")[
    ["n_cells", "logMAE_unwtd", "logMAE_acre_wtd", "x_off_acre_wtd"]]

print(f"ABLATION LADDER -- Natural acres, held-out season_year >= {TEST_START}\n")
print(ladder.round(4).to_string())

base = ladder.loc[f"persistence (k={K})", "x_off_acre_wtd"]
best = ladder["x_off_acre_wtd"].min()
print(f"\nacre-weighted: persistence floor {base:.1f}x off -> best rung {best:.1f}x off")
print("The metric that matters is x_off_acre_wtd -- that is where the floor failed.")

## Where this branch goes next

The external-covariate layer deferred since W2 is now sourced and wired (TerraClimate, CONUS + AK,
pre-season only). Remaining rungs:

- **Read the ablation result honestly.** If the acre-weighted error does *not* come down, that is a
  finding, not a failure — it would say pre-season seasonal-mean dryness is too coarse a signal for
  megafire years, and the next move is finer temporal resolution (monthly or daily extremes such as
  consecutive dry days) rather than more covariates.
- **Lag sensitivity.** The window is 3 pre-season months by default. Whether 6 or 12 months does
  better is an open empirical question — `tc.build(..., lag_months=N)` regenerates the table, and
  antecedent-wetness effects on fuel are known to operate over longer horizons than drought itself.
- **Revisit grain.** Level III may be coarse for mitigation siting; whether to go finer is the
  deferred spatial question — test only if the coarse product proves useful.
- **Concentration forecast.** Beyond per-cell acres, predict the *concentration* itself (HHI /
  top-region) so the planner knows whether next season's burn will pile into a few regions or
  spread — a different, coarser mitigation-planning signal.
- **Fuel load remains unrepresented**, deliberately. LANDFIRE's vintage structure makes it unusable
  as a time-varying feature on a 1992–2020 panel (see the covariate section above). If a
  cross-sectional fuel descriptor is wanted later, it enters as a *static* per-region feature, not
  a per-year one.